# Tokenization

Tokenizer Training

In [ ]:
import sentencepiece as spm
import os

dataset_file = "data/multilang_train.txt"
model_prefix = "spm_multilang"
vocab_size = 50000
num_threads = 6        # Safe for 8-core M2
input_sentence_size = 10000000  # 10M lines sampled for better coverage
progress_interval = 100000      # Show progress every 100k lines

# Estimate file size in MB
file_size_mb = os.path.getsize(dataset_file) / (1024 * 1024)
print(f"📦 Dataset size: {file_size_mb:.2f} MB")
print(f"🧵 Using {num_threads} threads, vocab size {vocab_size}, input_sentence_size {input_sentence_size}\n")

# Train tokenizer
spm.SentencePieceTrainer.train(
    input=dataset_file,
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    character_coverage=1.0,       # Full coverage for English, Hindi, Bengali
    model_type="unigram",          # Unigram is better for multilingual LM
    num_threads=num_threads,
    train_extremely_large_corpus=True,   # Stream file, don't load full dataset
    input_sentence_size=input_sentence_size,
    shuffle_input_sentence=True
)

print(f"\n✅ Tokenizer training completed!")
print(f"Files saved as: {model_prefix}.model & {model_prefix}.vocab")

📦 Dataset size: 30357.46 MB
🧵 Using 6 threads, vocab size 50000, input_sentence_size 10000000


✅ Tokenizer training completed!
Files saved as: spm_multilang.model & spm_multilang.vocab


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: data/multilang_train.txt
  input_format: 
  model_prefix: spm_multilang
  model_type: UNIGRAM
  vocab_size: 50000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 10000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 6
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential

Encoding train/val/test datasets into token IDs

In [3]:
import sentencepiece as spm
import os
from tqdm import tqdm

# Load your trained tokenizer
sp = spm.SentencePieceProcessor(model_file="spm_multilang.model")

# Dataset files
input_files = {
    "data/multilang_train.txt": "data/multilang_train.tok",
    "data/multilang_val.txt": "data/multilang_val.tok",
    "data/multilang_test.txt": "data/multilang_test.tok"
}

# Checkpoint parameters
CHECKPOINT_LINES = 500_000  # Save after every 500k lines

for in_file, out_file in input_files.items():
    print(f"\n📄 Encoding {in_file} → {out_file} with checkpoints every {CHECKPOINT_LINES} lines")
    
    # Determine where to resume if checkpoint exists
    checkpoint_file = out_file + ".checkpoint"
    start_line = 0
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            start_line = int(f.read().strip())
        print(f"🔄 Resuming from line {start_line}")
    
    # Open input/output files
    with open(in_file, "r", encoding="utf-8") as fin, \
            open(out_file, "a", encoding="utf-8") as fout:  # append mode to resume
    
        for line_idx, line in enumerate(tqdm(fin, desc=f"Encoding {in_file}", unit=" lines")):
            if line_idx < start_line:
                continue  # skip already processed lines
            
            line = line.strip()
            if not line:
                continue
            
            token_ids = sp.encode(line, out_type=int)
            fout.write(" ".join(map(str, token_ids)) + "\n")
            
            # Checkpoint every N lines
            if (line_idx + 1) % CHECKPOINT_LINES == 0:
                with open(checkpoint_file, "w") as cp:
                    cp.write(str(line_idx + 1))
                print(f"💾 Checkpoint saved at line {line_idx + 1}")
    
    # Remove checkpoint when finished
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
    print(f"✅ Finished encoding {in_file}. Saved to {out_file}")


📄 Encoding data/multilang_train.txt → data/multilang_train.tok with checkpoints every 500000 lines


Encoding data/multilang_train.txt: 508818 lines [00:06, 76490.52 lines/s]

💾 Checkpoint saved at line 500000


Encoding data/multilang_train.txt: 1010465 lines [00:13, 69197.98 lines/s]

💾 Checkpoint saved at line 1000000


...

💾 Checkpoint saved at line 8000000


Encoding data/multilang_test.txt: 8504714 lines [02:16, 44020.68 lines/s]

💾 Checkpoint saved at line 8500000


Encoding data/multilang_test.txt: 8764189 lines [02:20, 62392.23 lines/s]

✅ Finished encoding data/multilang_test.txt. Saved to data/multilang_test.tok


# Evaluation

- Make sure your vocabulary actually covers all three languages: English, Hindi, Bengali.

In [26]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor(model_file="spm_multilang.model")

# Encode a sample
sample = "Hello दुनिया! এই আমি using a encoded multilingual tokenizer."
tokens = sp.encode(sample, out_type=str)
print("Tokens:", tokens)

Tokens: ['▁Hello', '▁दुनिया', '!', '▁এই', '▁আমি', '▁using', '▁a', '▁en', 'code', 'd', '▁multi', 'lingual', '▁token', 'izer', '.']


Observations:
- English words are mostly subword-split
- Hindi (दुनिया) and Bengali (এই, আমি) words appear as full tokens
- Punctuation is kept separate
- Multilingual coverage is working — tokenizer is recognizing English, Devanagari, Bengali scripts

Encode/decode test

In [27]:
text = "Hello दुनिया, এই আমি"
ids = sp.encode(text)
print(ids)
decoded = sp.decode(ids)
print(decoded)

[22057, 1072, 3, 121, 441]
Hello दुनिया, এই আমি


In [4]:
# For Hindi & Bengali specific testing
sp = spm.SentencePieceProcessor(model_file="spm_multilang.model")

samples = [
    "यह एक बहुत अच्छा उदाहरण है।",   # Hindi
    "আমি বাংলা ভাষা ভালোবাসি।"       # Bengali
]

for text in samples:
    tokens = sp.encode(text, out_type=str)
    decoded = sp.decode(sp.encode(text))
    print("Original:", text)
    print("Tokens  :", tokens)
    print("Decoded :", decoded)
    print()

Original: यह एक बहुत अच्छा उदाहरण है।
Tokens  : ['▁यह', '▁एक', '▁बहुत', '▁अच्छा', '▁उदाहरण', '▁है', '।']
Decoded : यह एक बहुत अच्छा उदाहरण है।

Original: আমি বাংলা ভাষা ভালোবাসি।
Tokens  : ['▁আমি', '▁বাংলা', '▁ভাষা', '▁ভালোবাসি', '।']
Decoded : আমি বাংলা ভাষা ভালোবাসি।



Quick check for validating .tok encoded datasets - Checking how many unknown tokens (`<unk>`) appear in this encoded datasets.

In [29]:
from collections import Counter

# -----------------------------
# Configuration
# -----------------------------
input_files = {
    "train": "data/multilang_train.tok",
    "val": "data/multilang_val.tok",
    "test": "data/multilang_test.tok"
}

print("📊 Checking <unk> tokens in encoded datasets...\n")

# -----------------------------
# Evaluation
# -----------------------------
for split_name, file_path in input_files.items():
    total_tokens = 0
    unk_tokens = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            # Each line is a sequence of token IDs (space-separated)
            ids = list(map(int, line.split()))
            total_tokens += len(ids)
            unk_tokens += ids.count(0)   # <unk> = 0 in SentencePiece

            if line_num % 1_000_000 == 0:
                print(f"[{split_name}] Processed {line_num:,} lines...")

    unk_percentage = (unk_tokens / max(total_tokens, 1)) * 100
    print(f"✅ {split_name} dataset")
    print(f"   Total tokens : {total_tokens:,}")
    print(f"   <unk> tokens : {unk_tokens:,}")
    print(f"   <unk> %      : {unk_percentage:.4f}%\n")

📊 Checking <unk> tokens in encoded datasets...

[train] Processed 1,000,000 lines...
[train] Processed 2,000,000 lines...
[train] Processed 3,000,000 lines...
[train] Processed 4,000,000 lines...
[train] Processed 5,000,000 lines...
[train] Processed 6,000,000 lines...
[train] Processed 7,000,000 lines...
[train] Processed 8,000,000 lines...
[train] Processed 9,000,000 lines...
[train] Processed 10,000,000 lines...
[train] Processed 11,000,000 lines...
[train] Processed 12,000,000 lines...
[train] Processed 13,000,000 lines...
[train] Processed 14,000,000 lines...
[train] Processed 15,000,000 lines...
[train] Processed 16,000,000 lines...
[train] Processed 17,000,000 lines...
[train] Processed 18,000,000 lines...
[train] Processed 19,000,000 lines...
[train] Processed 20,000,000 lines...
[train] Processed 21,000,000 lines...
[train] Processed 22,000,000 lines...
[train] Processed 23,000,000 lines...
[train] Processed 24,000,000 lines...
[train] Processed 25,000,000 lines...
[train] Pro

Train:

`<unk>%`: 0.0002%

Test:

`<unk>%`: 0.0002%

Val:

`<unk>%`: 0.0002%

Analysis:
- <unk> percentage is extremely low: 0.0002% across all splits
- This means almost all tokens are covered by your vocabulary
- Tokenizer’s vocab size is sufficient for English + Hindi + Bengali

Final evalluation: Computing  percentage, average tokens per sentence, and subword frequency counter

In [1]:
import sentencepiece as spm
from collections import Counter

# -----------------------------
# Configuration
# -----------------------------
tokenizer_model = "spm_multilang.model"
datasets = {
    "train": "data/multilang_train.txt",
    "val": "data/multilang_val.txt",
    "test": "data/multilang_test.txt"
}
print_sample_count = 10  # Number of sample sentences to show

# -----------------------------
# Load tokenizer
# -----------------------------
sp = spm.SentencePieceProcessor(model_file=tokenizer_model)
print(f"✅ Loaded tokenizer: {tokenizer_model}")

# -----------------------------
# Evaluation
# -----------------------------
for split_name, file_path in datasets.items():
    print(f"\n📊 Evaluating {split_name} dataset: {file_path}")
    
    total_sentences = 0
    total_tokens = 0
    unk_count = 0
    subword_counter = Counter()
    sample_enc_dec = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            # Encode as IDs and subwords
            ids = sp.encode(line)
            tokens = sp.encode(line, out_type=str)

            total_sentences += 1
            total_tokens += len(tokens)
            unk_count += tokens.count("<unk>")
            subword_counter.update(tokens)

            # Collect sample sentences
            if len(sample_enc_dec) < print_sample_count:
                decoded = sp.decode(ids)
                sample_enc_dec.append((line, tokens, decoded))

            # Show progress every 1M lines
            if line_num % 1_000_000 == 0:
                print(f"Processed {line_num:,} lines...")

    # Compute stats
    avg_tokens_per_sentence = total_tokens / max(total_sentences, 1)
    unk_percentage = (unk_count / max(total_tokens, 1)) * 100

    print(f"\n✅ Evaluation complete for {split_name} dataset")
    print(f"Total sentences processed  : {total_sentences:,}")
    print(f"Total tokens processed     : {total_tokens:,}")
    print(f"<unk> token percentage     : {unk_percentage:.4f}%")
    print(f"Average tokens per sentence: {avg_tokens_per_sentence:.2f}")
    print(f"Top 20 most frequent subwords: {subword_counter.most_common(20)}")

    print(f"\nSample encoded/decoded sentences (first {print_sample_count}):")
    for orig, tokens, decoded in sample_enc_dec:
        print(f"Original : {orig}")
        print(f"Tokens   : {tokens}")
        print(f"Decoded  : {decoded}\n")

✅ Loaded tokenizer: spm_multilang.model

📊 Evaluating train dataset: data/multilang_train.txt
Processed 1,000,000 lines...
Processed 2,000,000 lines...
Processed 3,000,000 lines...
Processed 4,000,000 lines...
Processed 5,000,000 lines...
Processed 6,000,000 lines...
Processed 7,000,000 lines...
Processed 8,000,000 lines...
Processed 9,000,000 lines...
Processed 10,000,000 lines...
Processed 11,000,000 lines...
Processed 12,000,000 lines...
Processed 13,000,000 lines...
Processed 14,000,000 lines...
Processed 15,000,000 lines...
Processed 16,000,000 lines...
Processed 17,000,000 lines...
Processed 18,000,000 lines...
Processed 19,000,000 lines...
Processed 20,000,000 lines...
Processed 21,000,000 lines...
Processed 22,000,000 lines...
Processed 23,000,000 lines...
Processed 24,000,000 lines...
Processed 25,000,000 lines...
Processed 26,000,000 lines...
Processed 27,000,000 lines...
Processed 28,000,000 lines...
Processed 29,000,000 lines...
Processed 30,000,000 lines...
Processed 31,00

### Analysis

1. Token Coverage
- `<unk>` token percentage: 0.0000% across train, val, and test.

2. Average Tokens per Sentence
- Train: ~25.23
- Val/Test: ~25.22
- Indicates dataset splits were uniformly prepared.

3. Top Subwords
- Learned language-specific frequent subwords across all three scripts.
- Punctuation is well captured.

4. Sample Encoded/Decoded Sentences
- The round-trip decoding matches original text perfectly
- Confirms tokenizer generalizes well across English morphology, Hindi postpositions, Bengali punctuation.